**STEP 1: DATA LOADING**

In [210]:
import pandas as pd

In [211]:
df = pd.read_csv("ambucast_data.csv")

In [212]:
df.columns

Index(['date', 'hour', 'zone_name', 'call_count', 'PM2.5', 'PM10', 'AQI',
       'temperature', 'humidity', 'population_density', 'elderly_pct',
       'day_of_week', 'is_peak_hour'],
      dtype='object')

In [213]:
# # Convert to classification target
# threshold = df["call_count"].quantile(0.60)

# df["high_demand"] = (df["call_count"] >= threshold).astype(int)

**⚙️ STEP 2: FEATURE ENGINEERING**

In [214]:
import numpy as np

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

**📈 Rolling feature**

In [215]:
df = df.sort_values(["zone_name", "date", "hour"])

df["rolling_calls_7"] = (
    df.groupby("zone_name")["call_count"]
    .transform(lambda x: x.rolling(24 * 7, min_periods=1).mean())
)

**📊 Lag feature**

In [216]:
df["lag_24h"] = (
    df.groupby("zone_name")["call_count"]
    .shift(24)
)

In [217]:
df = df.dropna()

**⚙️ STEP 3: FEATURES SELECTION**

In [218]:
features = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "PM2.5", "PM10", "AQI",
    "temperature", "humidity",
    "population_density", "elderly_pct",
    "rolling_calls_7", "lag_24h"
]

x = df[features]
y = df["call_count"]

**STEP 4: TRAIN / TEST SPLIT (TIME-BASED)**

In [219]:
split_date = "2016-07-01"

train = df[df["date"] < split_date]
test  = df[df["date"] >= split_date]

x_train = train[features]
y_train = train["call_count"]

x_test = test[features]
y_test = test["call_count"]

**⚙️ STEP 5: TRAIN XGBOOST**

In [220]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [221]:
model.fit(x_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [222]:
y_pred = model.predict(x_test)


**STEP 6: EVALUATION METRICS**

In [223]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 1.2529584169387817
MSE: 3.0902655124664307
RMSE: 3.0902655124664307
R2 Score: 0.9600533246994019


**SAMPLE PREDICTIONS**

In [224]:
def predict_hotspots(input_df):

    # Predict call counts
    preds = model.predict(input_df[features])

    # Fix negatives
    preds = np.clip(preds, 0, None)

    # Convert to risk score (0–100)
    risk_score = (preds / preds.max()) * 100

    # Attach results
    input_df = input_df.copy()
    input_df["predicted_calls"] = preds.round().astype(int)

    input_df["risk_score"] = risk_score.round(2)

    return input_df[["zone_name", "predicted_calls", "risk_score"]]

In [225]:
sample = df.sample(20)

predict_hotspots(sample)

,zone_name,predicted_calls,risk_score
133747,North East,6,26.540001
77513,New Delhi,1,3.450000
4697,New Delhi,0,1.120000
72864,West,20,87.550003
15371,New Delhi,1,3.340000
33263,North,7,28.870001
16394,North,7,31.450001
30153,New Delhi,1,3.750000
104940,West,19,85.019997
105589,South West,18,80.129997


**STEP 7: EXPORT**

In [226]:
import pickle

with open("hotspotcast.pkl", "wb") as f:
     pickle.dump(model, f)